# DMS practice

Runnable form of the syllabus in `readme.md`. Run the cells from top to bottom.

Every write uses the three lab spaces derived from `PREFIX` (`dms_lab_schema`, `dms_lab_types`, `dms_lab_data`). The last cell deletes them.

Credentials come from the environment, or from `redonline-cdf-ingest/.env` when a variable is unset. The secret is never printed.


## Before you start

Steps:

1. Install `cognite-sdk` in this Jupyter kernel.
2. Provide `CDF_PROJECT`, `CDF_CLUSTER`, `CDF_TENANT_ID`, `CDF_CLIENT_ID`, and `CDF_CLIENT_SECRET`.
3. Keep `PREFIX` unique to you if several people share the project.
4. Run the cell. Done when it prints the project name and a space-list page without raising.


In [ ]:
from pathlib import Path
import os

from cognite.client import CogniteClient, ClientConfig
from cognite.client.credentials import OAuthClientCredentials
from cognite.client.data_classes.aggregations import Avg, Count
from cognite.client.data_classes.data_modeling import (
    BTreeIndexApply,
    ContainerApply,
    ContainerId,
    ContainerPropertyApply,
    DataModelApply,
    DirectRelation,
    DirectRelationReference,
    EdgeApply,
    Float64,
    InvolvedContainers,
    InvolvedViews,
    MappedPropertyApply,
    MultiEdgeConnectionApply,
    NodeApply,
    NodeOrEdgeData,
    NodeResultSetExpression,
    Query,
    RequiresConstraintApply,
    Select,
    SourceSelector,
    SpaceApply,
    Text,
    ViewApply,
    ViewId,
    EdgeResultSetExpression,
)
from cognite.client.data_classes import filters
from cognite.client.exceptions import CogniteAPIError


def load_env_file(path: Path) -> None:
    """Fill missing environment variables from a local .env file."""
    if not path.is_file():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


for candidate in (
    Path.cwd() / "redonline-cdf-ingest" / ".env",
    Path.cwd().parent / "redonline-cdf-ingest" / ".env",
    Path.cwd().parent / "Cursor" / "redonline-cdf-ingest" / ".env",
):
    load_env_file(candidate)

required = [
    "CDF_PROJECT",
    "CDF_TENANT_ID",
    "CDF_CLIENT_ID",
    "CDF_CLIENT_SECRET",
]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing environment variables: " + ", ".join(missing))

# Lab namespace. Change this if you share the CDF project with someone else.
PREFIX = "dms_lab"
SCHEMA = f"{PREFIX}_schema"
TYPES = f"{PREFIX}_types"
DATA = f"{PREFIX}_data"
VERSION = "1"

cluster = os.environ.get("CDF_CLUSTER", "az-eastus-1")
base_url = os.environ.get("CDF_BASE_URL", f"https://{cluster}.cognitedata.com")
tenant_id = os.environ["CDF_TENANT_ID"]
token_url = os.environ.get(
    "CDF_TOKEN_URL",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
)

client = CogniteClient(
    ClientConfig(
        client_name="dms-playbook",
        project=os.environ["CDF_PROJECT"],
        base_url=base_url,
        credentials=OAuthClientCredentials(
            token_url=token_url,
            client_id=os.environ["CDF_CLIENT_ID"],
            client_secret=os.environ["CDF_CLIENT_SECRET"],
            scopes=[f"{base_url}/.default"],
        ),
    )
)

probe = client.data_modeling.spaces.list(limit=1)
print("project:", client.config.project)
print("base_url:", base_url)
print("lab spaces:", SCHEMA, TYPES, DATA)
print("space list page size:", len(probe))


def view_props(instance, view_id: ViewId) -> dict:
    """Return the property dict stored for one view on a node or edge."""
    if instance.properties is None or view_id not in instance.properties:
        return {}
    return dict(instance.properties[view_id])


## 1. Property graph

Steps:

1. Read the three nodes and two links in the cell.
2. Run the cell and confirm the area link is a direct relation and the flow is an edge.
3. Say which link can store a comment and be traversed from the valve back to the pump.

Done when the cell prints both checks as passed.


In [ ]:
# Identifiers used by every later cell.
AREA_ID = "area-1"
PUMP_ID = "pump-42"
VALVE_ID = "valve-1"
EDGE_ID = "pump-42-to-valve-1"
TYPE_AREA = "area"
TYPE_PUMP = "pump"
TYPE_VALVE = "valve"
TYPE_FLOWS_TO = "flows-to"

graph = {
    "nodes": [
        {"space": DATA, "external_id": AREA_ID, "type": (TYPES, TYPE_AREA)},
        {"space": DATA, "external_id": PUMP_ID, "type": (TYPES, TYPE_PUMP)},
        {"space": DATA, "external_id": VALVE_ID, "type": (TYPES, TYPE_VALVE)},
    ],
    "direct_relations": [
        {
            "from": (DATA, PUMP_ID),
            "property": "area",
            "to": (DATA, AREA_ID),
            "why": "foreign key, no properties of its own",
        },
        {
            "from": (DATA, VALVE_ID),
            "property": "area",
            "to": (DATA, AREA_ID),
            "why": "foreign key, no properties of its own",
        },
    ],
    "edges": [
        {
            "space": DATA,
            "external_id": EDGE_ID,
            "type": (TYPES, TYPE_FLOWS_TO),
            "start_node": (DATA, PUMP_ID),
            "end_node": (DATA, VALVE_ID),
            "properties": {"comment": "Directs liquid to valve 1"},
            "why": "the link has its own comment and can be walked inwards",
        }
    ],
}

assert graph["direct_relations"][0]["property"] == "area"
assert graph["edges"][0]["properties"]["comment"]
assert graph["edges"][0]["type"][1] == TYPE_FLOWS_TO
print("nodes:", [node["external_id"] for node in graph["nodes"]])
print("direct relation:", graph["direct_relations"][0]["from"][1], "->", graph["direct_relations"][0]["to"][1])
print("edge:", graph["edges"][0]["start_node"][1], "->", graph["edges"][0]["end_node"][1])
print("checks passed")


## 2. Spaces

Steps:

1. Apply the schema space, the types space, and the data space.
2. Retrieve each space and print its name.
3. Keep schema writes in the schema space and instance writes in the data space.

Done when all three retrieves return a space.


In [ ]:
client.data_modeling.spaces.apply(
    [
        SpaceApply(
            SCHEMA,
            name="DMS lab schema",
            description="Containers, views, and the EquipmentLab data model.",
        ),
        SpaceApply(
            TYPES,
            name="DMS lab types",
            description="Type nodes for area, pump, valve, and flows-to.",
        ),
        SpaceApply(
            DATA,
            name="DMS lab data",
            description="Pump, valve, area, and flow instances.",
        ),
    ]
)

for space in (SCHEMA, TYPES, DATA):
    loaded = client.data_modeling.spaces.retrieve(space)
    print(loaded.space, "|", loaded.name)


## 3. Containers

Steps:

1. Apply `Area` and `Equipment` for nodes. `Equipment.area` is a direct relation that requires data in `Area`.
2. Index `manufacturer` (cursorable) and `area`. Bound indexed text with `max_text_size`.
3. Apply `Pump` with a `requires` constraint on `Equipment`.
4. Apply `Flow` with `used_for="edge"`.
5. Retrieve `Equipment` and confirm both index names are present. State may still be `pending`.

Done when the four containers retrieve and `Pump` lists the `requireEquipment` constraint.


In [ ]:
area_container = ContainerId(SCHEMA, "Area")
equipment_container = ContainerId(SCHEMA, "Equipment")
pump_container = ContainerId(SCHEMA, "Pump")
flow_container = ContainerId(SCHEMA, "Flow")

client.data_modeling.containers.apply(
    [
        ContainerApply(
            space=SCHEMA,
            external_id="Area",
            name="Area",
            used_for="node",
            properties={
                "name": ContainerPropertyApply(
                    Text(max_text_size=256),
                    nullable=False,
                    description="Area name",
                ),
            },
        ),
        ContainerApply(
            space=SCHEMA,
            external_id="Equipment",
            name="Equipment",
            used_for="node",
            properties={
                "manufacturer": ContainerPropertyApply(
                    Text(max_text_size=256),
                    nullable=False,
                    description="Manufacturer name",
                ),
                "area": ContainerPropertyApply(
                    DirectRelation(container=area_container),
                    nullable=True,
                    description="Direct relation to the area node",
                ),
            },
            indexes={
                "manufacturer": BTreeIndexApply(
                    properties=["manufacturer"],
                    cursorable=True,
                ),
                "area": BTreeIndexApply(properties=["area"]),
            },
        ),
        ContainerApply(
            space=SCHEMA,
            external_id="Flow",
            name="Flow",
            used_for="edge",
            properties={
                "comment": ContainerPropertyApply(
                    Text(max_text_size=1024),
                    nullable=True,
                    description="Comment stored on the flow edge",
                ),
            },
        ),
    ]
)

client.data_modeling.containers.apply(
    ContainerApply(
        space=SCHEMA,
        external_id="Pump",
        name="Pump",
        used_for="node",
        properties={
            "maxPressure": ContainerPropertyApply(
                Float64(),
                nullable=False,
                description="Maximum pressure",
            ),
        },
        constraints={
            "requireEquipment": RequiresConstraintApply(equipment_container),
        },
    )
)

equipment = client.data_modeling.containers.retrieve((SCHEMA, "Equipment"))
pump = client.data_modeling.containers.retrieve((SCHEMA, "Pump"))
print("Equipment indexes:", sorted(equipment.indexes))
print("Pump constraints:", sorted(pump.constraints))
for external_id in ("Area", "Equipment", "Pump", "Flow"):
    loaded = client.data_modeling.containers.retrieve((SCHEMA, external_id))
    print(loaded.space, loaded.external_id, "used_for=", loaded.used_for)


## 4. Views

Steps:

1. Map container properties into version `1` views. Query and ingestion go through these views.
2. Set the source of `Equipment.area` to the `Area` view.
3. Apply `Flow` before `Pump`, because the edge connection points at the `Flow` view.
4. Apply `Pump` so it implements `Equipment`, maps `maxPressure`, and declares outwards `flowsTo`.
5. Retrieve `Pump` with inherited properties and read the property identifiers.

Done when `Pump` implements `Equipment` and lists `manufacturer`, `area`, `maxPressure`, and `flowsTo`.


In [ ]:
area_view = ViewId(SCHEMA, "Area", VERSION)
equipment_view = ViewId(SCHEMA, "Equipment", VERSION)
pump_view = ViewId(SCHEMA, "Pump", VERSION)
flow_view = ViewId(SCHEMA, "Flow", VERSION)

client.data_modeling.views.apply(
    [
        ViewApply(
            space=SCHEMA,
            external_id="Area",
            version=VERSION,
            name="Area",
            properties={
                "name": MappedPropertyApply(area_container, "name"),
            },
        ),
        ViewApply(
            space=SCHEMA,
            external_id="Equipment",
            version=VERSION,
            name="Equipment",
            properties={
                "manufacturer": MappedPropertyApply(equipment_container, "manufacturer"),
                "area": MappedPropertyApply(
                    equipment_container,
                    "area",
                    source=area_view,
                ),
            },
        ),
        ViewApply(
            space=SCHEMA,
            external_id="Flow",
            version=VERSION,
            name="Flow",
            properties={
                "comment": MappedPropertyApply(flow_container, "comment"),
            },
        ),
    ]
)

client.data_modeling.views.apply(
    ViewApply(
        space=SCHEMA,
        external_id="Pump",
        version=VERSION,
        name="Pump",
        implements=[equipment_view],
        properties={
            "maxPressure": MappedPropertyApply(pump_container, "maxPressure"),
            "flowsTo": MultiEdgeConnectionApply(
                type=DirectRelationReference(TYPES, TYPE_FLOWS_TO),
                source=equipment_view,
                direction="outwards",
                edge_source=flow_view,
                name="Flows to",
            ),
        },
    )
)

loaded_views = client.data_modeling.views.retrieve(
    (SCHEMA, "Pump", VERSION),
    include_inherited_properties=True,
)
loaded = loaded_views[0]
print("implements:", [(item.space, item.external_id, item.version) for item in loaded.implements or []])
print("properties:", sorted(loaded.properties))


## 5. Data models

Steps:

1. Group `Area`, `Equipment`, `Pump`, and `Flow` into `EquipmentLab` version `1`.
2. Retrieve the model and print each view identifier.

Done when the retrieved model lists those four views.


In [ ]:
client.data_modeling.data_models.apply(
    DataModelApply(
        space=SCHEMA,
        external_id="EquipmentLab",
        version=VERSION,
        name="Equipment lab",
        description="Practice model for the DMS playbook.",
        views=[area_view, equipment_view, pump_view, flow_view],
    )
)

model = client.data_modeling.data_models.retrieve((SCHEMA, "EquipmentLab", VERSION))[0]
print(model.space, model.external_id, model.version)
for item in model.views:
    print(" view:", item.space, item.external_id, item.version)


## 6. System schemas

Steps:

1. List global spaces (`include_global=True`) and keep the ones whose names start with `cdf_`.
2. List global data models in `cdf_cdm`.
3. Open one view whose external id contains `Asset` and print its property identifiers.
4. Leave the lab model unchanged. This cell only reads.

Done when you can point at a global asset view and name three of its properties. If `cdf_cdm` is absent, the cell prints the global spaces it could see.


In [ ]:
spaces = client.data_modeling.spaces.list(limit=None, include_global=True)
global_spaces = sorted(space.space for space in spaces if space.space.startswith("cdf_"))
print("global spaces:", global_spaces)

if "cdf_cdm" not in global_spaces:
    print("cdf_cdm is not visible with this token. Read the space list above and stop this cell.")
else:
    models = client.data_modeling.data_models.list(
        space="cdf_cdm",
        limit=20,
        include_global=True,
    )
    print("cdf_cdm models:")
    for item in models:
        print(" ", item.external_id, item.version, "views", len(item.views))

    views = client.data_modeling.views.list(
        space="cdf_cdm",
        limit=None,
        include_global=True,
    )
    asset_views = [item for item in views if "Asset" in item.external_id]
    print("asset views:", [item.external_id for item in asset_views])
    if asset_views:
        sample = asset_views[0]
        print("sample view:", sample.external_id, sample.version)
        print("properties:", sorted(sample.properties)[:12])


## 7. Nodes

Steps:

1. Apply the type nodes `area`, `pump`, `valve`, and `flows-to` in the types space.
2. Apply `area-1`, `valve-1`, and `pump-42` through their views. Leave `area` empty until the next section.
3. Retrieve `pump-42` through `Pump` and read `version`, `manufacturer`, and `maxPressure`.
4. List data-space nodes whose `manufacturer` equals `Acme`.

Done when the retrieve prints version `1` and the list contains the pump and the valve.


In [ ]:
type_nodes = [
    NodeApply(TYPES, external_id)
    for external_id in (TYPE_AREA, TYPE_PUMP, TYPE_VALVE, TYPE_FLOWS_TO)
]
client.data_modeling.instances.apply(type_nodes)

client.data_modeling.instances.apply(
    [
        NodeApply(
            DATA,
            AREA_ID,
            type=(TYPES, TYPE_AREA),
            sources=[NodeOrEdgeData(area_view, {"name": "Unit 100"})],
        ),
        NodeApply(
            DATA,
            VALVE_ID,
            type=(TYPES, TYPE_VALVE),
            sources=[
                NodeOrEdgeData(
                    equipment_view,
                    {"manufacturer": "Acme"},
                )
            ],
        ),
        NodeApply(
            DATA,
            PUMP_ID,
            type=(TYPES, TYPE_PUMP),
            sources=[
                NodeOrEdgeData(
                    pump_view,
                    {"manufacturer": "Acme", "maxPressure": 1.2},
                )
            ],
        ),
    ],
    auto_create_direct_relations=False,
)

pump_node = client.data_modeling.instances.retrieve(
    nodes=(DATA, PUMP_ID),
    sources=[pump_view],
).nodes[0]
pump_properties = view_props(pump_node, pump_view)
print("version:", pump_node.version)
print("manufacturer:", pump_properties.get("manufacturer"))
print("maxPressure:", pump_properties.get("maxPressure"))

acme = client.data_modeling.instances.list(
    instance_type="node",
    sources=[equipment_view],
    space=DATA,
    filter=filters.Equals(equipment_view.as_property_ref("manufacturer"), "Acme"),
    limit=25,
)
print("Acme nodes:", [node.external_id for node in acme])


## 8. Direct relations and edges

Steps:

1. Merge the `area` direct relation onto the pump and the valve. Both already have `manufacturer`, and merge keeps it.
2. Apply one edge from `pump-42` to `valve-1` with type `flows-to` and a comment written through `Flow`.
3. Retrieve the edge and read its start, end, type, and comment.

Done when the pump's `area` external id is `area-1` and the edge comment is `Directs liquid to valve 1`.


In [ ]:
area_ref = {"space": DATA, "externalId": AREA_ID}

client.data_modeling.instances.apply(
    [
        NodeApply(
            DATA,
            VALVE_ID,
            sources=[NodeOrEdgeData(equipment_view, {"area": area_ref})],
        ),
        NodeApply(
            DATA,
            PUMP_ID,
            sources=[NodeOrEdgeData(pump_view, {"area": area_ref})],
        ),
    ],
    replace=False,
    auto_create_direct_relations=False,
)

client.data_modeling.instances.apply(
    edges=[
        EdgeApply(
            space=DATA,
            external_id=EDGE_ID,
            type=(TYPES, TYPE_FLOWS_TO),
            start_node=(DATA, PUMP_ID),
            end_node=(DATA, VALVE_ID),
            sources=[
                NodeOrEdgeData(
                    flow_view,
                    {"comment": "Directs liquid to valve 1"},
                )
            ],
        )
    ],
    auto_create_start_nodes=False,
    auto_create_end_nodes=False,
)

pump_node = client.data_modeling.instances.retrieve(
    nodes=(DATA, PUMP_ID),
    sources=[pump_view],
).nodes[0]
print("pump area:", view_props(pump_node, pump_view).get("area"))

flow = client.data_modeling.instances.retrieve(
    edges=(DATA, EDGE_ID),
    sources=[flow_view],
).edges[0]
print("start:", flow.start_node)
print("end:", flow.end_node)
print("type:", flow.type)
print("comment:", view_props(flow, flow_view).get("comment"))


## 9. Apply, merge, replace, and delete

Steps:

1. Merge `maxPressure = 1.5` with `replace=False`. Read the version and confirm `manufacturer` is still `Acme`.
2. Send the full property set with `replace=True` and manufacturer `Acme Industries`. Read the new version.
3. Apply `scratch-node`, delete it, and confirm a retrieve returns no node.

Done when the pump version has increased twice and the scratch node is absent.


In [ ]:
before = client.data_modeling.instances.retrieve(
    nodes=(DATA, PUMP_ID),
    sources=[pump_view],
).nodes[0]
print("before version:", before.version)

client.data_modeling.instances.apply(
    NodeApply(
        DATA,
        PUMP_ID,
        sources=[NodeOrEdgeData(pump_view, {"maxPressure": 1.5})],
    ),
    replace=False,
)
merged = client.data_modeling.instances.retrieve(
    nodes=(DATA, PUMP_ID),
    sources=[pump_view],
).nodes[0]
merged_props = view_props(merged, pump_view)
print("merged version:", merged.version)
print("merged manufacturer:", merged_props.get("manufacturer"))
print("merged maxPressure:", merged_props.get("maxPressure"))

client.data_modeling.instances.apply(
    NodeApply(
        DATA,
        PUMP_ID,
        sources=[
            NodeOrEdgeData(
                pump_view,
                {
                    "manufacturer": "Acme Industries",
                    "maxPressure": 1.5,
                    "area": {"space": DATA, "externalId": AREA_ID},
                },
            )
        ],
    ),
    replace=True,
    auto_create_direct_relations=False,
)
replaced = client.data_modeling.instances.retrieve(
    nodes=(DATA, PUMP_ID),
    sources=[pump_view],
).nodes[0]
print("replaced version:", replaced.version)
print("replaced manufacturer:", view_props(replaced, pump_view).get("manufacturer"))

client.data_modeling.instances.apply(NodeApply(DATA, "scratch-node"))
deleted = client.data_modeling.instances.delete(nodes=(DATA, "scratch-node"))
print("deleted:", [(node.space, node.external_id) for node in deleted.nodes])
leftover = client.data_modeling.instances.retrieve(nodes=(DATA, "scratch-node"))
print("scratch nodes left:", len(leftover.nodes))


## 10. Query

Steps:

1. Filter pumps with the query parameter `maker` set to `Acme Industries`.
2. Chain an outwards edge result set limited to type `flows-to`.
3. Chain the nodes at the end of those edges and select manufacturer on both sides.

Done when the result contains `pump-42` and `valve-1`.


In [ ]:
query = Query(
    with_={
        "pumps": NodeResultSetExpression(
            filter=filters.Equals(
                pump_view.as_property_ref("manufacturer"),
                filters.ParameterValue("maker"),
            ),
            limit=25,
        ),
        "flows": EdgeResultSetExpression(
            from_="pumps",
            direction="outwards",
            filter=filters.Equals(
                ["edge", "type"],
                {"space": TYPES, "externalId": TYPE_FLOWS_TO},
            ),
            max_distance=1,
        ),
        "valves": NodeResultSetExpression(from_="flows", limit=25),
    },
    select={
        "pumps": Select(
            [SourceSelector(pump_view, ["manufacturer", "maxPressure", "area"])],
            limit=25,
        ),
        "valves": Select(
            [SourceSelector(equipment_view, ["manufacturer", "area"])],
            limit=25,
        ),
    },
    parameters={"maker": "Acme Industries"},
)

result = client.data_modeling.instances.query(query)
print("pumps:", [node.external_id for node in result["pumps"]])
print("valves:", [node.external_id for node in result["valves"]])
for node in result["pumps"]:
    print(" pump properties:", view_props(node, pump_view))


## 11. Search and aggregation

Steps:

1. Search `Equipment.manufacturer` for `Acme` inside the data space.
2. Count equipment nodes grouped by `manufacturer`.
3. Average `maxPressure` on the `Pump` view.

Done when the search returns the pump and the valve, the grouped count shows `Acme Industries` and `Acme`, and the average prints a number.


In [ ]:
found = client.data_modeling.instances.search(
    equipment_view,
    query="Acme",
    properties=["manufacturer"],
    space=DATA,
    limit=25,
)
print("search:", [node.external_id for node in found])

counts = client.data_modeling.instances.aggregate(
    equipment_view,
    Count("externalId"),
    group_by="manufacturer",
    space=DATA,
    limit=25,
)
print("counts:")
for row in counts:
    print(" ", row.group, row.aggregates)

average = client.data_modeling.instances.aggregate(
    pump_view,
    Avg("maxPressure"),
    space=DATA,
)
print("average maxPressure:", average)


## 12. Access control and inspection

Steps:

1. Inspect the token and print `dataModelsAcl` and `dataModelInstancesAcl` actions and scopes.
2. Read the btree index state on `Equipment`. `pending` means the background build is still running.
3. Read the `requires` constraint state on `Pump`.
4. Inspect `pump-42` and list the involved views and containers.

Done when you can name the capability that lets this token write the schema space and the capability that lets it write the data space.


In [ ]:
token = client.iam.token.inspect()
print("project on token:", getattr(token, "projects", None))
for capability in token.capabilities:
    dumped = capability.dump()
    for name, spec in dumped.items():
        if "model" in name.lower():
            print(name)
            print("  actions:", spec.get("actions"))
            print("  scope:", spec.get("scope"))

equipment = client.data_modeling.containers.retrieve((SCHEMA, "Equipment"))
print("Equipment index state:")
for name, index in equipment.indexes.items():
    print(f"  {name}: {getattr(index, 'state', None)}")

pump = client.data_modeling.containers.retrieve((SCHEMA, "Pump"))
print("Pump constraint state:")
for name, constraint in pump.constraints.items():
    print(f"  {name}: {getattr(constraint, 'state', None)} require={constraint.require}")

report = client.data_modeling.instances.inspect(
    nodes=(DATA, PUMP_ID),
    involved_views=InvolvedViews(all_versions=True),
    involved_containers=InvolvedContainers(),
)
print(report)


## Reset the lab

Steps, in this order:

1. Delete edges, then nodes, in the data space.
2. Delete the type nodes. A type node stays while any instance still points at it.
3. Delete the data model, then views `Pump`, `Flow`, `Equipment`, `Area`, then the containers in that order.
4. Delete the data, types, and schema spaces.
5. Run the notebook from the top if you want to practice again.

Done when the three space retrieves print `None`.


In [ ]:
def delete_quietly(action, label: str) -> None:
    try:
        action()
        print("deleted", label)
    except CogniteAPIError as exc:
        print("skip", label, exc.code, exc.message)


edges = client.data_modeling.instances.list(
    instance_type="edge",
    space=DATA,
    limit=None,
)
nodes = client.data_modeling.instances.list(
    instance_type="node",
    space=DATA,
    limit=None,
)
if edges:
    client.data_modeling.instances.delete(edges=edges.as_ids())
if nodes:
    client.data_modeling.instances.delete(nodes=nodes.as_ids())
print("data edges:", len(edges), "data nodes:", len(nodes))

type_nodes = client.data_modeling.instances.list(
    instance_type="node",
    space=TYPES,
    limit=None,
)
if type_nodes:
    client.data_modeling.instances.delete(nodes=type_nodes.as_ids())
print("type nodes:", len(type_nodes))

delete_quietly(
    lambda: client.data_modeling.data_models.delete((SCHEMA, "EquipmentLab", VERSION)),
    "data model EquipmentLab",
)
for external_id in ("Pump", "Flow", "Equipment", "Area"):
    delete_quietly(
        lambda external_id=external_id: client.data_modeling.views.delete(
            (SCHEMA, external_id, VERSION)
        ),
        f"view {external_id}",
    )
for external_id in ("Pump", "Flow", "Equipment", "Area"):
    delete_quietly(
        lambda external_id=external_id: client.data_modeling.containers.delete(
            (SCHEMA, external_id)
        ),
        f"container {external_id}",
    )
for space in (DATA, TYPES, SCHEMA):
    delete_quietly(
        lambda space=space: client.data_modeling.spaces.delete(space),
        f"space {space}",
    )

for space in (SCHEMA, TYPES, DATA):
    print(space, client.data_modeling.spaces.retrieve(space))
